# Проект-1

**Автор, группа:** Чухрова Злата, БАЦРФ-261.

**Гипотеза:** канал performance наименее окупаемый из-за высокой стоимости привлечения клиентов по сравнению с другими каналами.

**Единица анализа:** зарегистрированный клиент, включая клиента без подписки и оплат.

В расчёт входят только пользователи, использующие ПРАЙМ полгода (180 дней). Если по клиентам месяца ещё нет 180 дней истории, весь месяц и его маркетинговые расходы исключаются.

Более подробно можно прочитать в приложенной аналитической записке.

## 1. Настройки и подключение

Подключаемся к базе:

In [ ]:
import json
import sys

import numpy as np
import pandas as pd
from sqlalchemy import text

# get_engine() живёт в вашем репозитории, в src/prime/config.py.
# Она читает строку подключения из .env — поэтому пароля здесь нет.
try:
    from prime.config import get_engine
except ModuleNotFoundError:
    raise ModuleNotFoundError(
        "Пакет prime не найден: ноутбук запущен не в окружении prime-monitor.\n"
        f"Сейчас Python отсюда: {sys.executable}\n"
        "JupyterLab: закройте его, перейдите в корень prime-monitor и выполните uv run jupyter lab.\n"
        "VS Code: Select Kernel справа сверху -> .venv из папки prime-monitor."
    ) from None

engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу."""
    try:
        with engine.connect() as conn:
            # Моменты времени считаем в UTC: так они у всех одинаковые.
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


print("подключились как:", q("select current_user as login").iloc[0, 0])

Импортируем инструменты для графиков и вывода результатов.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

Задаём папку для сохранения таблиц и графиков.

In [ ]:
ROOT = Path.cwd()
ROOT = ROOT / "project01" if (ROOT / "project01").is_dir() else ROOT
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
FIGURES = ROOT / "reports/figures"
FIGURES.mkdir(parents=True, exist_ok=True)

Параметры, которые можно изменять в данной ячейке:

- `cohort_start`, `cohort_end` — начало и конец периода **регистрации**; конец не включается. Для февраля–июля можно указать `2026-02-01` и `2026-08-01`. Обе даты должны быть первыми числами месяца.
- `horizon_days` — одинаковое число дней после регистрации каждого клиента.
- `data_cutoff` — дата отсечения платежной истории, не включается. `None` использует конец доступной истории; будущая дата ограничивается фактическими данными БД.
- `cohort_split` — граница двух групп когорт. `None` делит реально доступные месяцы пополам. При одном месяце декомпозиция изменения пропускается.
- `comparison_channels` — произвольный список источников сравнения. Например, добавьте `"referral"` или `"crm"`; `[]` оставляет только основной источник. Повторы удаляются.

In [ ]:
analysis_config = {
    "question": "Почему выбранный источник не покрывает расходы?",
    "hypothesis": "Высокая стоимость регистрации при сопоставимых показателях оплат.",
    "cohort_start": "2026-01-01", "cohort_end": "2026-03-01",
    "horizon_days": 180, "data_cutoff": None, "cohort_split": None,
    "focus_channel": "performance", "comparison_channels": ["social", "partner_telecom"],
    "metric": "Подписочные поступления за H дней / расходы на привлечение",
}

Настраиваем компактный вывод чисел и единое оформление графиков.

In [ ]:
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
pd.set_option("display.max_columns", 10)
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False})
channel_names = {"performance": "Performance", "partner_telecom": "Partner telecom", "social": "Social",
                 "referral": "Реферальный", "organic": "Органика", "crm": "CRM"}
focus_name = channel_names.get(analysis_config["focus_channel"], analysis_config["focus_channel"])

Узнаём фактические границы платежной истории. Последний календарный день считается доступным до следующей полуночи UTC; наличие внутренних пропусков проверяется отдельно.

In [ ]:
bounds = q("SELECT min(paid_at) AS first_payment, max(paid_at) AS last_payment FROM prime.payments").iloc[0]
first_payment = pd.to_datetime(bounds.first_payment, utc=True)
last_payment = pd.to_datetime(bounds.last_payment, utc=True)
available_end = last_payment.normalize() + pd.Timedelta(days=1)
configured_cutoff = analysis_config["data_cutoff"]
data_end = min(pd.Timestamp(configured_cutoff, tz="UTC"), available_end) if configured_cutoff else available_end

Преобразуем пользовательские даты и горизонт в объекты для расчёта.

In [ ]:
cohort_start = pd.Timestamp(analysis_config["cohort_start"], tz="UTC")
requested_end = pd.Timestamp(analysis_config["cohort_end"], tz="UTC")
horizon = int(analysis_config["horizon_days"])
if horizon <= 0 or cohort_start >= requested_end:
    raise ValueError("Нужны положительный горизонт и начало периода раньше конца.")
if cohort_start.day != 1 or requested_end.day != 1:
    raise ValueError("Для месячных когорт укажите первые числа месяцев.")

Вычисляем последний полностью созревший месяц. Фильтрация применяется до загрузки и клиентов, и расходов.

In [ ]:
mature_until = data_end - pd.Timedelta(days=horizon)
mature_month_end = mature_until.replace(day=1).normalize()
used_end = min(requested_end, mature_month_end)
if cohort_start < first_payment.normalize():
    raise ValueError("Для начала выбранного периода в базе нет полной истории платежей.")
if used_end <= cohort_start:
    raise ValueError(f"Нет полных созревших месяцев. Начало должно быть раньше {mature_month_end.date()}.")

Выбираем границу двух групп. При единственном доступном месяце межпериодное разложение не рассчитывается.

In [ ]:
month_starts = pd.date_range(cohort_start, used_end, freq="MS", inclusive="left")
configured_split = analysis_config["cohort_split"]
auto_split = month_starts[len(month_starts) // 2] if len(month_starts) > 1 else None
split_at = pd.Timestamp(configured_split, tz="UTC") if configured_split else auto_split
if split_at is not None and (not cohort_start < split_at < used_end or split_at.day != 1):
    raise ValueError("cohort_split должен быть первым числом внутри реально используемого периода; либо задайте None.")

Названия групп выводим из дат, чтобы они менялись вместе с конфигурацией.

In [ ]:
def period_label(left, right) -> str:
    first = left.strftime("%Y-%m")
    last = (right - pd.Timedelta(days=1)).strftime("%Y-%m")
    return first if first == last else f"{first} — {last}"
period_order = [period_label(cohort_start, used_end)]
if split_at is not None:
    period_order = [period_label(cohort_start, split_at), period_label(split_at, used_end)]
recent_label = period_order[-1]

Перед расчётом показываем запрошенный и фактически используемый периоды. Исключённые месяцы не попадают ни в регистрации, ни в маркетинговые расходы.

In [ ]:
excluded_months = pd.date_range(used_end, requested_end, freq="MS", inclusive="left").strftime("%Y-%m").tolist()
selection = {"Запрошены регистрации": f"[{cohort_start.date()}, {requested_end.date()})",
             "Использованы регистрации": f"[{cohort_start.date()}, {used_end.date()})",
             "Горизонт после регистрации": f"{horizon} дней", "Данные строго до": str(data_end.date())}
display(pd.Series(selection, name="Период расчёта"))
print("Исключённые незрелые месяцы:", ", ".join(excluded_months) or "нет")

Все запросы получают эффективный период и единую дату отсечения.

In [ ]:
params = {"start": cohort_start.isoformat(), "end": used_end.isoformat(),
          "days": horizon, "as_of": data_end.isoformat()}
print("Группы сравнения:", period_order)
print("Предельный момент регистрации для полного окна:", mature_until)

Получаем всех зарегистрированных клиентов, включая клиентов без оплат.

In [ ]:
clients_sql = """
SELECT client_id, registered_at, acquisition_channel AS channel
FROM prime.clients
WHERE registered_at >= CAST(:start AS timestamptz)
  AND registered_at < CAST(:end AS timestamptz)
"""

Для платежей выбираем владельца подписки. Суммы загружаем в целых копейках.

In [ ]:
payments_sql = """
SELECT c.client_id, p.payment_id, p.subscription_id, p.paid_at,
       (100 * p.amount)::bigint AS amount_kopecks, s.tariff_id
FROM prime.clients c
JOIN prime.subscriptions s ON s.owner_client_id = c.client_id
JOIN prime.payments p ON p.subscription_id = s.subscription_id
"""

Оставляем успешные платежи в одинаковом окне после регистрации каждого клиента.

In [ ]:
payments_sql += """
WHERE c.registered_at >= CAST(:start AS timestamptz)
  AND c.registered_at < CAST(:end AS timestamptz)
  AND p.status = 'success' AND p.paid_at >= c.registered_at
  AND p.paid_at < CAST(:as_of AS timestamptz)
  AND p.paid_at < c.registered_at + CAST(:days AS int) * interval '1 day'
"""

Расходы сначала суммируем по месяцу и каналу. Это не позволит размножить их при соединении с клиентскими итогами.

In [ ]:
marketing_sql = """
SELECT to_char(spend_date, 'YYYY-MM') AS month, channel,
       (100 * sum(amount))::bigint AS spend_kopecks
FROM prime.marketing_spend
WHERE spend_date >= CAST(:start AS date) AND spend_date < CAST(:end AS date)
GROUP BY 1, 2
"""

Загружаем три таблицы и закрываем подключение. Дальше работаем в pandas.

In [ ]:
clients = q(clients_sql, **params)
payments = q(payments_sql, **params)
marketing = q(marketing_sql, **params)
engine.dispose()
pd.Series({"Клиенты": len(clients), "Успешные платежи": len(payments), "Месяц × канал расходов": len(marketing)})

## 2. Витрина клиентов

Приводим даты к UTC и проверяем уникальность ключей.

In [ ]:
clients["registered_at"] = pd.to_datetime(clients.registered_at, utc=True)
payments["paid_at"] = pd.to_datetime(payments.paid_at, utc=True)
clients["window_end"] = clients.registered_at + pd.Timedelta(days=horizon)
assert clients.client_id.is_unique and payments.payment_id.is_unique
assert clients.window_end.le(data_end).all()
clients[["registered_at", "window_end"]].dtypes.to_frame("Тип")

В базе известен пропуск платежей 14–15 апреля 2026 года. Показываем, сколько выбранных клиентских окон его пересекает; эти суммы не восстанавливаем.

In [ ]:
gap_start = pd.Timestamp("2026-04-14", tz="UTC")
gap_end = pd.Timestamp("2026-04-16", tz="UTC")
affected = clients.registered_at.lt(gap_end) & clients.window_end.gt(gap_start)
print(f"Окон с апрельским пропуском: {affected.sum():,} из {len(clients):,}")

Отсутствующий канал сохраняем отдельной категорией. Создаём месяц и группу когорт.

In [ ]:
print(f"Клиентов без канала: {clients.channel.isna().sum():,}")
clients["channel"] = clients.channel.fillna("Без канала")
clients["month"] = clients.registered_at.dt.strftime("%Y-%m")
clients["period"] = recent_label
if split_at is not None:
    clients["period"] = np.where(clients.registered_at < split_at, period_order[0], period_order[1])

Присоединяем дату регистрации к оплатам с проверкой связи «много к одному».

In [ ]:
original_sum = payments.amount_kopecks.sum()
paid = payments.merge(clients[["client_id", "registered_at"]], on="client_id", validate="m:1")
assert paid.amount_kopecks.sum() == original_sum
paid = paid.sort_values(["client_id", "paid_at", "payment_id"])

Отмечаем близкие повторные списания одинаковой суммы. Ниже используем их только для проверки чувствительности.

In [ ]:
previous_at = paid.groupby(["subscription_id", "amount_kopecks"]).paid_at.shift()
close = paid.paid_at.sub(previous_at).lt(pd.Timedelta(minutes=20))
paid["close_count"] = close.astype(int)
paid["close_kopecks"] = paid.amount_kopecks.where(close, 0)

Группируем оплаты до одного клиента. Первый тариф берём после сортировки по времени оплаты.

In [ ]:
by_client = paid.groupby("client_id").agg(
    payments=("payment_id", "size"), receipts_kopecks=("amount_kopecks", "sum"),
    first_paid_at=("paid_at", "min"), first_tariff=("tariff_id", "first"),
    close_count=("close_count", "sum"), close_kopecks=("close_kopecks", "sum"),
)
by_client.head()

Левое соединение сохраняет неплательщиков. Нулями заполняем только счётчики и суммы.

In [ ]:
cohort = clients.merge(by_client, on="client_id", how="left", validate="1:1")
zero_columns = ["payments", "receipts_kopecks", "close_count", "close_kopecks"]
cohort[zero_columns] = cohort[zero_columns].fillna(0).astype("int64")
cohort["paying"] = cohort.payments.gt(0)
assert cohort.receipts_kopecks.sum() == original_sum

Считаем итоги по месяцу регистрации и каналу. В конверсии учитываются все регистрации.

In [ ]:
monthly = cohort.groupby(["month", "period", "channel"], as_index=False).agg(
    registrations=("client_id", "size"), payers=("paying", "sum"),
    payments=("payments", "sum"), receipts_kopecks=("receipts_kopecks", "sum"),
)
monthly = monthly.merge(marketing, on=["month", "channel"], how="left", validate="1:1")
monthly["receipts"], monthly["spend"] = monthly.receipts_kopecks / 100, monthly.spend_kopecks / 100

Состав источников сравнения берём из конфигурации. Выводим доступные варианты и проверяем опечатки; повторяющиеся названия удаляем.

In [ ]:
focus = list(dict.fromkeys([analysis_config["focus_channel"], *analysis_config["comparison_channels"]]))
available_sources = sorted(monthly.loc[monthly.spend.gt(0), "channel"].unique())
unknown_sources = [source for source in focus if source not in available_sources]
if unknown_sources:
    raise ValueError(f"Источники без данных или расходов: {unknown_sources}. Доступны: {available_sources}")
print("Доступные источники:", ", ".join(available_sources))
print("Выбраны:", ", ".join(focus))

## 3. Показатели каналов

**K = V × F × P / C**, где V — доля плательщиков среди регистраций, F — оплат на плательщика, P — средний платёж, C — расходы на регистрацию.

K = 1 означает равенство подписочных платежей за N выбранных дней и приписанных маркетинговых расходов. В показатель не входят остальные доходы и затраты компании. Расходы сопоставляются с регистрациями того же месяца и канала; это правило расчёта, а не доказанная причинная атрибуция рекламы.

Одинаково рассчитываем метрики для месячных и объединённых итогов.

In [ ]:
def add_metrics(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    result["V"] = result.payers / result.registrations
    result["F"] = result.payments / result.payers
    result["P"] = result.receipts / result.payments
    result["C"] = result.spend / result.registrations
    result["per_client"] = result.receipts / result.registrations
    result["K"] = result.receipts / result.spend
    return result

Сначала складываем суммы и числители, затем считаем отношения. Месячные коэффициенты не усредняем.

In [ ]:
totals = ["registrations", "payers", "payments", "receipts", "spend"]
summary = monthly.groupby(["period", "channel"])[totals].sum(min_count=1).reset_index()
summary = add_metrics(summary)
monthly = add_metrics(monthly)
recent = summary.loc[summary.period.eq(recent_label)].set_index("channel")
print("Основная таблица: группа", recent_label)
if recent.reindex(focus).payers.fillna(0).eq(0).any():
    raise ValueError("В итоговой группе нет плательщиков у одного из выбранных источников; измените период, горизонт или список сравнения.")

In [ ]:
table = recent[["registrations", "V", "F", "P", "C", "per_client", "K"]].rename(columns={
    "registrations": "Регистрации", "V": "Конверсия", "F": "Оплат на плательщика", "P": "Средний платёж, ₽",
    "C": "Стоимость регистрации, ₽", "per_client": "Платежи на регистрацию, ₽", "K": "Покрытие"})
table.style.format({"Регистрации": "{:,.0f}", "Конверсия": "{:.1%}", "Оплат на плательщика": "{:.2f}",
    "Средний платёж, ₽": "{:,.0f}", "Стоимость регистрации, ₽": "{:,.0f}", "Платежи на регистрацию, ₽": "{:,.0f}", "Покрытие": "{:.1%}"}, na_rep="—")

Для дальнейшего исследования были выбраны каналы Social	и Partner telecom как наиболее близкие к рассматриваемым

In [ ]:
table = recent.loc[focus, ["registrations", "V", "F", "P", "C", "per_client", "K"]].rename(index=channel_names, columns={
    "registrations": "Регистрации", "V": "Конверсия", "F": "Оплат на плательщика", "P": "Средний платёж, ₽",
    "C": "Стоимость регистрации, ₽", "per_client": "Платежи на регистрацию, ₽", "K": "Покрытие"})
table.style.format({"Регистрации": "{:,.0f}", "Конверсия": "{:.1%}", "Оплат на плательщика": "{:.2f}",
    "Средний платёж, ₽": "{:,.0f}", "Стоимость регистрации, ₽": "{:,.0f}", "Платежи на регистрацию, ₽": "{:,.0f}", "Покрытие": "{:.1%}"}, na_rep="—")

Подготавливаем подписи каналов для графика стоимости и поступлений.

In [ ]:
known = recent.loc[focus].dropna(subset=["C"])
limit = max(known.C.max(), known.per_client.max()) * 1.12
offsets = {"organic": (-15, 15), "crm": (0, -20), "referral": (0, 15),
           "partner_telecom": (-35, 15), "social": (4, -20), "performance": (-65, 15)}

**График 1.** Каждая точка — канал. Положение относительно диагонали показывает покрытие приписанных расходов подписочными платежами.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5), layout="constrained")
ax.plot([0, limit], [0, limit], "--", color="gray", label="Покрытие 100%")
for channel, row in known.iterrows():
    ax.scatter(row.C, row.per_client, s=75, color="#c16d45" if channel == analysis_config["focus_channel"] else "#387695")
    ax.annotate(channel_names.get(channel, channel), (row.C, row.per_client), xytext=offsets.get(channel, (5, 10)), textcoords="offset points")
ax.set(xlim=(0, limit), ylim=(0, limit), title="Стоимость и платежи", xlabel="Стоимость регистрации, ₽", ylabel=f"Платежи на регистрацию за {horizon} дней, ₽")
ax.legend(frameon=False)
fig.savefig(FIGURES / "performance_01_channels.png", dpi=160)
plt.show()

Выделяем четыре фактора покрытия для основного источника и выбранного списка сравнения.

In [ ]:
factor_names = {"V": "Конверсия, %", "F": "Оплат на плательщика",
                "P": "Средний платёж, ₽", "C": "Стоимость регистрации, ₽"}
recent.loc[focus, ["V", "F", "P", "C"]]

**График 2.** Сопоставляем показатели оплат и стоимость регистрации в итоговой группе когорт.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 6), layout="constrained")
for ax, factor in zip(axes.flat, factor_names):
    values = recent.loc[focus, factor] * (100 if factor == "V" else 1)
    bars = ax.bar([channel_names.get(c, c) for c in focus], values, color=["#c16d45"] + ["#7cacc2"] * (len(focus) - 1))
    ax.bar_label(bars, fmt="%.1f" if factor in ["V", "F"] else "%.0f", padding=3, fontsize=9)
    ax.set(title=factor_names[factor], ylim=(0, values.max() * 1.2))
fig.savefig(FIGURES / "performance_02_metrics.png", dpi=160)
plt.show()

## 4. Факторное разложение

Анализируем, почему изменился уровень покрытия маркетинговых расходов платежами подпискимежду двумя периодами. Для этого последовательно оцениваем влияние четырёх факторов: конверсии в оплату (V), среднего числа платежей на плательщика (F), среднего размера платежа (P) и стоимости привлечения клиента (C).

Выбираем два периода только для performance и записываем формулу покрытия.

In [ ]:
performance = summary.loc[summary.channel.eq(analysis_config["focus_channel"])].set_index("period").reindex(period_order)
def coverage(row) -> float:
    return row["V"] * row["F"] * row["P"] / row["C"]
if len(period_order) == 2:
    before, after = performance.loc[period_order[0]], performance.loc[period_order[1]]
performance[["V", "F", "P", "C", "K"]]

Заменяем по одному фактору и проверяем совпадение суммы вкладов с изменением покрытия.

In [ ]:
effects = {}
if len(period_order) == 2:
    current = before.copy()
    for factor in factor_names:
        old_value = coverage(current)
        current[factor] = after[factor]
        effects[factor] = 100 * (coverage(current) - old_value)
    assert np.isclose(sum(effects.values()), 100 * (after.K - before.K))

Показываем вклады факторов только при наличии двух групп.

In [ ]:
if effects:
    display(pd.Series(effects).rename(index=factor_names).rename("Вклад в изменение K, п.п.").to_frame())
else:
    print("Декомпозиция пропущена: доступна одна месячная когорта.")

## 5. Проверка по месяцам и тарифам

Проверяем выбранные созревшие месяцы и влияние долей первого оплаченного тарифа. Общие веса рассчитываются по плательщикам только выбранных источников. При изменении списка сравнения меняется и эта стандартизация.

Агрегируем платежи плательщиков по первому тарифу, месяцу и каналу.

In [ ]:
payers = cohort.loc[cohort.paying & cohort.channel.isin(focus)].copy()
tariff_values = payers.groupby(["month", "channel", "first_tariff"]).agg(
    payers=("client_id", "size"), receipts_kopecks=("receipts_kopecks", "sum"))
tariff_values["receipts"] = tariff_values.receipts_kopecks / 100
tariff_weights = payers.first_tariff.value_counts(normalize=True).sort_index()
tariff_weights.rename("Общая доля плательщиков").to_frame()

Применяем одинаковые доли тарифов, сохраняя наблюдаемые конверсию и стоимость регистрации каждого канала.

In [ ]:
tariff_values["weight"] = tariff_values.index.get_level_values("first_tariff").map(tariff_weights)
tariff_values["weighted_receipts"] = tariff_values.receipts / tariff_values.payers * tariff_values.weight
common_receipts = tariff_values.groupby(["month", "channel"]).weighted_receipts.sum()
stability = monthly.loc[monthly.channel.isin(focus)].set_index(["month", "channel"]).copy()
stability["common_K"] = stability.V * common_receipts / stability.C
view = stability[["C", "V", "K", "common_K"]].rename(index=channel_names, level="channel")
view = view.rename(columns={"C": "Стоимость, ₽", "V": "Конверсия", "K": "Покрытие", "common_K": "Общие доли тарифов"})
performance_series = stability.xs(analysis_config["focus_channel"], level="channel").sort_index()
view.style.format({"Стоимость, ₽": "{:,.0f}", "Конверсия": "{:.1%}", "Покрытие": "{:.1%}", "Общие доли тарифов": "{:.1%}"})

**График 3.** Линии показывают месячные когорты. Пунктир — основной источник при общих долях тарифов выбранных источников.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4), layout="constrained")
for channel in focus:
    series = stability.xs(channel, level="channel").sort_index()
    ax.plot(series.index, 100 * series.K, marker="o", label=channel_names.get(channel, channel))
ax.plot(performance_series.index, 100 * performance_series.common_K, "--", color="#333333", label=f"{focus_name}: общие доли тарифов")
ax.axhline(100, color="gray", linewidth=1)
ax.set(xlabel="Месяц регистрации", ylabel="Покрытие расходов, %", title="Покрытие по когортам")
ax.legend(frameon=False, fontsize=9)
fig.savefig(FIGURES / "performance_03_stability.png", dpi=160)
plt.show()

## 6. Пороги покрытия

Рассчитываем пороги для итоговой группы когорт при выбранном горизонте H. Каждый раз меняется один показатель, остальные сохраняются. Это арифметические условия покрытия, а не прогноз изменения рекламы.

Считаем необходимые стоимость регистрации, конверсию и число оплат для итоговой группы когорт.

In [ ]:
observed = recent.loc[analysis_config["focus_channel"]]
required_conversion = observed.C / (observed.F * observed.P)
required_frequency = observed.C / (observed.V * observed.P)
thresholds = pd.DataFrame([
    ["Стоимость регистрации", f"{observed.C:,.0f} ₽", f"{observed.per_client:,.0f} ₽"],
    ["Конверсия", f"{observed.V:.1%}", f"{required_conversion:.1%}"],
    ["Оплат на плательщика", f"{observed.F:.2f}", f"{required_frequency:.2f}"],
], columns=["Показатель", "Наблюдается", "Для покрытия 100%"])
thresholds

Берём ближайшие целые количества оплат ниже и выше расчётного порога. Эти сценарии показывают арифметику покрытия; достижимость проверяем по фактическим оплатам ниже.

In [ ]:
recent_payers = payers.loc[payers.period.eq(recent_label) & payers.channel.eq(analysis_config["focus_channel"])]
mean_payment = observed.P
scenario_counts = sorted({max(1, int(np.floor(required_frequency))), max(1, int(np.ceil(required_frequency)))})
scenarios = pd.DataFrame({"Оплат на плательщика": scenario_counts})
scenarios["Платежи на регистрацию, ₽"] = observed.V * scenarios["Оплат на плательщика"] * mean_payment
scenarios["Покрытие"] = scenarios["Платежи на регистрацию, ₽"] / observed.C
scenarios

**График 4.** Сравниваем факт и арифметические сценарии при прежних доле плательщиков и среднем платеже. Горизонт один и тот же; сценарий не утверждает, что столько оплат реально успеет произойти.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5), layout="constrained")
values = [observed.K, *scenarios.Покрытие]
labels = ["Факт", *[f"{count} оплат" for count in scenario_counts]]
bars = ax.barh(labels, np.array(values) * 100, color=["#c16d45"] + ["#387695"] * len(scenario_counts))
ax.bar_label(bars, fmt="%.1f%%", padding=4)
ax.axvline(100, color="gray", linestyle="--")
ax.set(xlim=(0, max(1, max(values)) * 115), xlabel="Покрытие расходов, %", title=f"{focus_name}: сценарии покрытия")
fig.savefig(FIGURES / "performance_04_scenarios.png", dpi=160)
plt.show()

## 7. Наблюдаемое число оплат

Проверяем, сколько плательщиков основного источника из итоговой группы когорт действительно достигли сценарных чисел оплат за N дней. Близкие повторные списания исключаются только в альтернативном расчёте.

Для каждого плательщика сравниваем фактическое число оплат с расчётом без кандидатов в повторы.

In [ ]:
recent_payers = recent_payers.copy()
recent_payers["payments_adjusted"] = recent_payers.payments - recent_payers.close_count
threshold_rows = []
for count in scenario_counts:
    threshold_rows.append({"Оплат не меньше": count, "Все оплаты": recent_payers.payments.ge(count).mean(),
                           "Без близких повторов": recent_payers.payments_adjusted.ge(count).mean()})
pd.DataFrame(threshold_rows).set_index("Оплат не меньше").style.format("{:.2%}")

Строим распределение числа оплат от всех плательщиков этой группы.

In [ ]:
distribution = pd.DataFrame({"Все оплаты": recent_payers.payments.value_counts(normalize=True),
                            "Без близких повторов": recent_payers.payments_adjusted.value_counts(normalize=True)})
distribution = distribution.fillna(0).sort_index().mul(100)
distribution

**График 5.** Столбцы показывают долю плательщиков с конкретным числом оплат. Каждый цвет суммируется до 100%.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")
distribution.plot.bar(ax=ax, color=["#7cacc2", "#387695"], rot=0)
ax.set(xlabel=f"Успешных оплат за {horizon} дней", ylabel="Плательщики, %", title=f"Число оплат: {focus_name}")
ax.legend(frameon=False)
fig.savefig(FIGURES / "performance_05_payment_counts.png", dpi=160)
plt.show()

Сохраняем параметры фактической выборки вместе с результатами, чтобы разные прогоны не перепутать.

In [ ]:
results = {"hypothesis": analysis_config["hypothesis"], "horizon_days": horizon,
    "requested_period": [analysis_config["cohort_start"], analysis_config["cohort_end"]],
    "used_period": [params["start"], params["end"]], "data_cutoff": data_end.isoformat(),
    "recent_period": recent_label, "cost_per_registration": float(observed.C),
    "receipts_per_registration": float(observed.per_client), "coverage": float(observed.K),
    "required_conversion": float(required_conversion), "required_payments": float(required_frequency),
    "channels": recent.loc[focus, ["V", "F", "P", "C", "per_client", "K"]].reset_index().to_dict("records"),
    "factor_contributions_pp": effects}

Файл содержит результаты последнего прогона. Графики также перезаписываются при смене конфигурации.

In [ ]:
output = ROOT / "data/processed/performance_project_results.json"
output.parent.mkdir(parents=True, exist_ok=True)
output.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")

## Вывод для выбранной конфигурации

Формулируем результат из текущих данных, без фиксированных дат и старых чисел.

In [ ]:
status = "покрывают" if observed.K >= 1 else "не покрывают"
display(Markdown(f"**{focus_name}, когорты {recent_label}.** За {horizon} дней подписочные поступления {status} приписанные расходы. "
    f"Стоимость регистрации — **{observed.C:,.0f} ₽**, поступления на регистрацию — **{observed.per_client:,.0f} ₽**, "
    f"покрытие — **{observed.K:.1%}**. Сравнение выполнено с источниками: "
    f"{', '.join(channel_names.get(c, c) for c in focus[1:]) or 'список сравнения пуст'}."))

Показываем условные пороги при сохранении остальных наблюдаемых показателей.

In [ ]:
display(Markdown(f"**Пороги для покрытия 100%.** Стоимость регистрации не выше **{observed.per_client:,.0f} ₽**, "
    f"либо конверсия **{required_conversion:.1%}**, либо **{required_frequency:.2f} оплаты** на плательщика. "
    "Конверсия выше 100% невозможна. Сценарные количества оплат нужно сопоставлять с наблюдаемым распределением, "
    "а изменение бюджета может менять число и состав клиентов."))